In [10]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score, confusion_matrix

df = pd.read_csv(r'C:\Users\bachu\Desktop\5thsem_OJT\project1_diabetes\data\processed\muhammed_bilal_p1w2_features.csv')


feature_cols = ["age","pregnancies","glucose","blood_pressure","skin_thickness","insulin","bmi","diabetes_pedigree","bmi_category","glucose_band"]
x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]

x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [11]:
candidates = {
    "LogReg": (
        "scaled",
        LogisticRegression(max_iter=1000)
    ),
    "Decision Tree": (
        "raw",
        DecisionTreeClassifier(max_depth=3, random_state=42)
    ),
    "KNN": (
        "scaled",
        KNeighborsClassifier(n_neighbors=25)
    ),
    "SVC balanced": (
        "scaled",
        SVC(class_weight="balanced", random_state=42)
    ),
    "RF balanced": (
        "raw",
        RandomForestClassifier(
            max_depth=5,
            class_weight="balanced",
            random_state=42
        )
    )
}

TARGET_RECALL, GUARD_F1 = 0.70, 0.50  # Week 5 decision

In [12]:
print("rows:", len(df), "| train:", len(x_train), "| test:", len(x_test))
print("diabetic in test set:", int(y_test.sum()))
print("features after get_dummies:", x.shape[1])
print("candidates:", list(candidates))
print("target: recall >=", TARGET_RECALL, "with F1 >=", GUARD_F1)

rows: 950 | train: 760 | test: 190
diabetic in test set: 55
features after get_dummies: 13
candidates: ['LogReg', 'Decision Tree', 'KNN', 'SVC balanced', 'RF balanced']
target: recall >= 0.7 with F1 >= 0.5


re- split the data and watch the score move

In [13]:
rows = []

for rs in [0, 1, 7, 42, 99]:
    Xa, Xb, ya, yb = train_test_split(
        x, y,
        test_size=0.2,
        random_state=rs,
        stratify=y
    )

    row = {"split": rs}

    for name, (kind, model) in candidates.items():
        m = clone(model)  # fresh copy

        if kind == "scaled":
            sc = StandardScaler().fit(Xa)
            m.fit(sc.transform(Xa), ya)
            pred = m.predict(sc.transform(Xb))

        else:
            m.fit(Xa, ya)
            pred = m.predict(Xb)
    else:
        m.fit(Xa,ya)
        pred = m.predict(Xb)
        row[name] = round(recall_score(yb,pred),3)
        rows.append(row)

        splits = pd.DataFrame(rows).set_index("split")
        print(splits)
        spread = (splits.max() - splits.min()).round(3)
        print("\nspread (best-worst):\n",spread.sort_values(ascending=False))


        # Add evaluation results here

       RF balanced
split             
0              0.6

spread (best-worst):
 RF balanced    0.0
dtype: float64
       RF balanced
split             
0            0.600
1            0.618

spread (best-worst):
 RF balanced    0.018
dtype: float64
       RF balanced
split             
0            0.600
1            0.618
7            0.582

spread (best-worst):
 RF balanced    0.036
dtype: float64
       RF balanced
split             
0            0.600
1            0.618
7            0.582
42           0.655

spread (best-worst):
 RF balanced    0.073
dtype: float64
       RF balanced
split             
0            0.600
1            0.618
7            0.582
42           0.655
99           0.655

spread (best-worst):
 RF balanced    0.073
dtype: float64


what k-fold cross validation 
rows = []

for rs in [0, 1, 7, 42, 99]:
    Xa, Xb, ya, yb = train_test_split(
        X, y,
        test_size=0.2,
        random_state=rs,
        stratify=y
    )

    row = {"split": rs}

    for name, (kind, model) in candidates.items():
        m = clone(model)  # fresh copy

        if kind == "scaled":
            sc = StandardScaler().fit(Xa)
            m.fit(sc.transform(Xa), ya)
            pred = m.predict(sc.transform(Xb))

        else:
            m.fit(Xa, ya)
            pred = m.predict(Xb)

        # Add evaluation results here

what is k-fold cross validation

K-Fold Cross-Validation is a technique in Machine Learning used to check how well a model performs by splitting the dataset into K equal or nearly equal parts (folds). The model is trained and validated multiple times using different parts of the data.
